In [48]:
import pandas as pd
from sklearn.model_selection import train_test_split

GETTING DATA

In [49]:
from pathlib import Path

cwd = Path.cwd()
data_folder = Path.cwd().parent / 'data_preprocessing' / 'processed_data'
train_data = data_folder / 'train_ready.parquet'

try:
    if not train_data.is_file():
        raise ValueError()
except Exception as e:
    print('file with .parquet not found. Please check in "ai/ai_tools/dataprocessing/processed_data".')
    print('Ensure that file under processed_data direcory is called *EXACTLY* "train_ready.parquet".')
    print('Else check path in this cell for path mistypes.')

SPLITTING

In [50]:
df = pd.read_parquet(train_data)

X = df['content']
Y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, 
                                                    Y , 
                                                    test_size=0.2, 
                                                    random_state=42)



SETTING UP TEXT EMBEDDOR

In [51]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

embeddor = TfidfVectorizer(stop_words='english')

LOADING CANDIDATE MODELS

In [52]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier

In [53]:
# some of these models take a while to train on CPU, so we choose models which can train relatively quickly
# and are known performers
Models = {
    'LinearSVC': Pipeline([('text_vectorizer', embeddor), ('model', LinearSVC(dual=False))]),
    'KNeighborsClassifier': Pipeline([('text_vectorizer', embeddor), ('model', KNeighborsClassifier(n_neighbors=5))]),
    'DecisionTreeClassifier': Pipeline([('text_vectorizer', embeddor), ('model', DecisionTreeClassifier(max_depth=20))]),
}

RUN TRAINING ON ALL MODELS

In [54]:
import time
import gc
from sklearn.metrics import classification_report, accuracy_score

def train_models(models : dict):
    
    training_times = {}
    inference_times = {}
    accuracies = {}
    f1_scores = {}
    precision_scores = {}
    
    for model_name, model in models.items():
    
        # train model, get training time
        start_time = time.time()
        model.fit(X_train, y_train)
        training_time = time.time() - start_time
        
        #inference time
        start_time = time.time()
        y_pred = model.predict(X_test)
        inference_time = time.time() - start_time
        
        # accuracy
        accuracy = accuracy_score(y_test, y_pred)
        
        # classification report
        class_report_dict = classification_report(y_test, y_pred, output_dict=True)
        
        # f1 score
        f1_avg_score = class_report_dict['weighted avg']['f1-score']
        precision = class_report_dict['weighted avg']['precision']
        
        # recording evaluation metrics
        training_times[model_name] = training_time
        inference_times[model_name] = inference_time
        accuracies[model_name] = accuracy
        f1_scores[model_name] = f1_avg_score
        precision_scores[model_name] = precision

        
        # printing evaluations
        print(f"Model: {model_name}")
        print(f"Training Time: {training_time:.6f} seconds")
        print(f"Prediction Time: {inference_time:.6f} seconds")
        print(f"Accuracy: {accuracy:.6f}")
        print("Classification Report:")
        print(classification_report(y_test, y_pred))
        print("="*60)
        
        # cleaning every model
        del model
        gc.collect()
        
        # pandas df of all classification reports for visualization
    results_df = pd.DataFrame({
    "Model": list(Models.keys()),
    "Accuracy": list(accuracies.values()),
    "Prediction Time (s)": list(inference_times.values()),
    "Training Time (s)": list(training_times.values()),
    "f1_scores" : list(f1_scores.values()),
    "precision_scores" : list(precision_scores.values())
    })
        
    return results_df


results_df = train_models(Models)

Model: LinearSVC
Training Time: 15.148219 seconds
Prediction Time: 0.599066 seconds
Accuracy: 0.696531
Classification Report:
                                            precision    recall  f1-score   support

           ABSTRACT_CLASSES_AND_INTERFACES       0.67      0.73      0.70       210
                                ALGORITHMS       0.48      0.48      0.48       194
                        ALGORITHM_ANALYSIS       0.71      0.64      0.67       231
                                 ARRAYS_1D       0.47      0.41      0.44       181
                                 ARRAYS_2D       0.64      0.69      0.66       188
                                 AVL_TREES       0.98      0.97      0.98       252
                       BINARY_SEARCH_TREES       0.71      0.79      0.75       193
                       CLASSES_AND_OBJECTS       0.55      0.46      0.50       221
                          CODING_STANDARDS       0.59      0.49      0.54       196
               COMPOSITION_AND_AG

c:\Users\johnd\miniconda3\envs\BERT_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\johnd\miniconda3\envs\BERT_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\johnd\miniconda3\envs\BERT_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", 

RESULT ANALYSIS

In [55]:
results_df.sort_values(by=['Accuracy'], ascending=False)

,Model,Accuracy,Prediction Time (s),Training Time (s),f1_scores,precision_scores
0,LinearSVC,0.696531,0.599066,15.148219,0.693186,0.694136
1,KNeighborsClassifier,0.475874,3.833582,2.881024,0.478103,0.508066
2,DecisionTreeClassifier,0.381032,0.550088,10.174363,0.378117,0.486486


WINNER GRID SEARCH TRAINING OPTIMIZATION

In [56]:
import joblib
from sklearn.model_selection import GridSearchCV

save_path = Path.cwd() / 'linearsvc_pipeline.joblib'

if not save_path.exists():
    
    # AI to train
    pipeline = Pipeline([
        ('text_vectorizer', TfidfVectorizer(stop_words="english",
                                            max_features=15000,
                                            min_df=3)), 
        ('model', LinearSVC(dual=False, 
                            class_weight='balanced')) # balance the influence of each class to address imbalance
    
    ])

    # 2. Define the Parameter Grid (using the step_name__parameter syntax)
    param_grid = {
        
        # TfIdf settings
        'text_vectorizer__max_features': [10000, 20000],
        'text_vectorizer__ngram_range': [(1,1), (1,2)],
        
        # LinearSVC Settings
        'model__C': [0.1, 1, 10],                      # regularization
        'model__tol': [1e-4, 1e-3]                     # Tolerance for stopping criteria
    }

    # grid search set up
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring='f1_macro',   
        cv=5,                 # 5 cross validations
        n_jobs=-1,            
        verbose=3 
    )

    # train
    grid_search.fit(X_train, y_train)

    best_model = grid_search.best_estimator_

    print("Best Parameters:", grid_search.best_params_)
    print("Best Score:", grid_search.best_score_)

    # Save the model using joblib
    joblib.dump(best_model, save_path,compress=9)
    print(f"Model saved to {save_path}")

else:
    print("Model already exists at this path.")

Fitting 5 folds for each of 24 candidates, totalling 120 fits
Best Parameters: {'model__C': 0.1, 'model__tol': 0.0001, 'text_vectorizer__max_features': 20000, 'text_vectorizer__ngram_range': (1, 2)}
Best Score: 0.7059817654596141
Model saved to c:\All University Materials\Project\ICT304-project\ai\ai_tools\nlp_classifier\linearsvc_pipeline.joblib


FUNCTIONALITY TESTING

In [57]:
grid_text_model = joblib.load(save_path)

In [ ]:
import re

abbreviation_map = {
    'bst': 'binary search tree',
    'dll': 'doubly linked list',
    'sll': 'singly linked list',
    'oop': 'object oriented programming',
    'dfs': 'depth first search',
    'bfs': 'breadth first search',
    'dp': 'dynamic programming',
    'tc': 'time complexity',
    'sc': 'space complexity',
    'll': 'linked list',
    'avl': 'avl tree',
    'ht': 'hash table',
    'pls': 'please',
    'plz': 'please',
    'idk': 'i do not know',
    'cant': 'cannot',
    'dont': 'do not',
    'doesnt': 'does not',
    'isnt': 'is not',
    'composition and inheritance': 'composition aggregation inheritance'
}

def preprocess(text):
    text = text.lower()
    for abbrev, expansion in abbreviation_map.items():
        text = re.sub(rf'\b{abbrev}\b', expansion, text)
    return text

In [47]:
    
topic_maps = {key : value for key, value in enumerate(df['topic'].value_counts().index)}

def predicts(text_classifier, topic_maps, questions, threshold=0.1, print_results=True):
    
    process_questions = [preprocess(q) for q in questions]
    
    scores = text_classifier.decision_function(process_questions)
    predictions = text_classifier.predict(process_questions)
    
    max_scores = scores.max(axis=1)
    
    results = []
    for score, pred in zip(max_scores, predictions):
        
        if score < threshold:
            results.append('OTHER')
        else:
            results.append(topic_maps[pred])
            
    if print_results:
        for question, label in zip(questions, results):
            print(f'Question: {question} -> Label: {label}')
        
    return results
        
        
        
predicts(
        text_classifier=grid_text_model,
        topic_maps=topic_maps,
        questions=["What is an array?", # basic functionality testing
                   "How can I design my classes better?", # vagueness
                   "Why are you trump?",# random test
                   "Are you a classifier because you are right? Or are you right because you are a classifier?",
                   "How can I do algorithm a",
                   "", # null
                   "How do I reverse a linked list?",
                    "What is the difference between a stack and a queue?",
                    "my bst keeps breaking when I delete a node",
                    "when should I use a hashmap vs a tree?",
                    "how do i make my sorting algorithm faster",
                    "whats the point of virtual functions",
                    "I don't understand pointers, please help?",
                    "how do avl trees balance themselves",
                    "whats the difference between composition and inheritance",
                    "my recursive function never stops",
                    "how do I read from a file in c++",
                    "what even is big o notation",
                    "when do I use an interface vs abstract class",
                    "my constructor isnt working",
                    "how do I traverse a graph"
                   ] 
        )

Question: What is an array? -> Label: ARRAYS_1D
Question: How can I design my classes better? -> Label: DESIGN_PATTERNS
Question: Why are you trump? -> Label: OTHER
Question: Are you a classifier because you are right? Or are you right because you are a classifier? -> Label: OTHER
Question: How can I do algorithm a -> Label: ALGORITHMS
Question:  -> Label: OTHER
Question: How do I reverse a linked list? -> Label: LINKED_LIST
Question: What is the difference between a stack and a queue? -> Label: STACKS
Question: my bst keeps breaking when I delete a node -> Label: BINARY_SEARCH_TREES
Question: when should I use a hashmap vs a tree? -> Label: HASH_TABLES
Question: how do i make my sorting algorithm faster -> Label: SORTING
Question: whats the point of virtual functions -> Label: POLYMORPHISM
Question: I don't understand pointers, please help? -> Label: POINTERS
Question: how do avl trees balance themselves -> Label: AVL_TREES
Question: whats the difference between composition and inheri

['ARRAYS_1D',
 'DESIGN_PATTERNS',
 'OTHER',
 'OTHER',
 'ALGORITHMS',
 'OTHER',
 'LINKED_LIST',
 'STACKS',
 'BINARY_SEARCH_TREES',
 'HASH_TABLES',
 'SORTING',
 'POLYMORPHISM',
 'POINTERS',
 'AVL_TREES',
 'COMPOSITION_AND_AGGREGATION',
 'CONTROL_FLOW',
 'FILE_IO',
 'ALGORITHM_ANALYSIS',
 'ABSTRACT_CLASSES_AND_INTERFACES',
 'CLASSES_AND_OBJECTS',
 'GRAPHS']

CONFUSION MATRIX DISPLAY

In [12]:
predictions = grid_text_model.predict(X_test)
true_values = y_test

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

classes = [topic_maps[index] for index in range(len(topic_maps))]
cm = confusion_matrix(true_values, predictions)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)

# GEMINI GENERATED CODE: 

# 1. Create a much larger figure and axis before plotting
fig, ax = plt.subplots(figsize=(16, 16)) 

# 2. Pass the axis to the plot and rotate the x-ticks
disp.plot(ax=ax, xticks_rotation='vertical') 

# 3. Add tight_layout so the long labels don't get cut off the edge of the screen
plt.tight_layout() 
plt.show()